In [2]:
!pip install yt-dlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.6 MB/s eta 0:00:00


In [1]:
import yt_dlp
from datetime import datetime
from pathlib import Path
import json
import re


class YouTubeChannelSearcher:
    """Classe pour rechercher des vidéos sur une chaîne YouTube"""
    
    def __init__(self, output_dir="videos_filtrees"):
        """
        Initialise le chercheur de vidéos
        
        Args:
            output_dir: Dossier de destination des vidéos
        """
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
    
    def obtenir_videos_chaine(self, url_chaine):
        """
        Récupère toutes les vidéos d'une chaîne YouTube
        
        Args:
            url_chaine: URL de la chaîne YouTube
                       (ex: https://www.youtube.com/@nom_chaine ou
                            https://www.youtube.com/c/nom_chaine ou
                            https://www.youtube.com/channel/CHANNEL_ID)
        
        Returns:
            list: Liste des informations de toutes les vidéos
        """
        ydl_opts = {
            'quiet': True,
            'no_warnings': True,
            'extract_flat': True,  # Ne télécharge pas, récupère juste les infos
            'force_generic_extractor': False,
        }
        
        try:
            print(f"🔍 Récupération des vidéos de la chaîne...")
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                # Ajouter /videos à l'URL pour obtenir toutes les vidéos
                if not url_chaine.endswith('/videos'):
                    url_chaine = url_chaine.rstrip('/') + '/videos'
                
                info = ydl.extract_info(url_chaine, download=False)
                
                if 'entries' in info:
                    videos = list(info['entries'])
                    print(f"✅ {len(videos)} vidéos trouvées sur la chaîne")
                    return videos
                else:
                    print("❌ Aucune vidéo trouvée")
                    return []
                    
        except Exception as e:
            print(f"❌ Erreur lors de la récupération des vidéos: {e}")
            return []
    
    def obtenir_details_video(self, video_id):
        """
        Récupère les détails complets d'une vidéo
        
        Args:
            video_id: ID de la vidéo YouTube
        
        Returns:
            dict: Informations détaillées de la vidéo
        """
        ydl_opts = {
            'quiet': True,
            'no_warnings': True,
        }
        
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                url = f"https://www.youtube.com/watch?v={video_id}"
                info = ydl.extract_info(url, download=False)
                return info
        except Exception as e:
            print(f"⚠️ Erreur pour la vidéo {video_id}: {e}")
            return None
    
    def filtrer_par_date(self, videos, date_debut=None, date_fin=None):
        """
        Filtre les vidéos par date de publication
        
        Args:
            videos: Liste des vidéos à filtrer
            date_debut: Date de début au format 'YYYY-MM-DD' ou 'YYYYMMDD'
            date_fin: Date de fin au format 'YYYY-MM-DD' ou 'YYYYMMDD'
        
        Returns:
            list: Vidéos filtrées
        """
        videos_filtrees = []
        
        # Convertir les dates en format YYYYMMDD
        if date_debut:
            date_debut = date_debut.replace('-', '')
        if date_fin:
            date_fin = date_fin.replace('-', '')
        
        print(f"🗓️ Filtrage par date...")
        if date_debut:
            print(f"   Début: {date_debut[:4]}-{date_debut[4:6]}-{date_debut[6:]}")
        if date_fin:
            print(f"   Fin: {date_fin[:4]}-{date_fin[4:6]}-{date_fin[6:]}")
        
        for video in videos:
            video_id = video.get('id')
            if not video_id:
                continue
            
            # Récupérer les détails pour avoir la date exacte
            details = self.obtenir_details_video(video_id)
            if not details:
                continue
            
            date_upload = details.get('upload_date', '')
            
            # Vérifier si la date est dans la plage
            if date_debut and date_upload < date_debut:
                continue
            if date_fin and date_upload > date_fin:
                continue
            
            videos_filtrees.append(details)
        
        print(f"✅ {len(videos_filtrees)} vidéos dans la plage de dates")
        return videos_filtrees
    
    def filtrer_par_mot_cle(self, videos, mot_cle, dans_titre=True, dans_description=True):
        """
        Filtre les vidéos contenant un mot-clé
        
        Args:
            videos: Liste des vidéos à filtrer
            mot_cle: Mot-clé à rechercher (insensible à la casse)
            dans_titre: Rechercher dans le titre
            dans_description: Rechercher dans la description
        
        Returns:
            list: Vidéos filtrées
        """
        videos_filtrees = []
        mot_cle_lower = mot_cle.lower()
        
        print(f"🔎 Filtrage par mot-clé: '{mot_cle}'")
        print(f"   Recherche dans: ", end="")
        recherche_dans = []
        if dans_titre:
            recherche_dans.append("titre")
        if dans_description:
            recherche_dans.append("description")
        print(", ".join(recherche_dans))
        
        for video in videos:
            titre = video.get('title', '').lower()
            description = video.get('description', '').lower()
            
            trouve = False
            if dans_titre and mot_cle_lower in titre:
                trouve = True
            if dans_description and mot_cle_lower in description:
                trouve = True
            
            if trouve:
                videos_filtrees.append(video)
        
        print(f"✅ {len(videos_filtrees)} vidéos contiennent le mot-clé")
        return videos_filtrees
    
    def rechercher_videos(self, url_chaine, mot_cle, date_debut=None, date_fin=None,
                         dans_titre=True, dans_description=True):
        """
        Recherche des vidéos sur une chaîne avec filtres
        
        Args:
            url_chaine: URL de la chaîne YouTube
            mot_cle: Mot-clé à rechercher
            date_debut: Date de début (format 'YYYY-MM-DD')
            date_fin: Date de fin (format 'YYYY-MM-DD')
            dans_titre: Rechercher dans le titre
            dans_description: Rechercher dans la description
        
        Returns:
            list: Liste des vidéos correspondant aux critères
        """
        print(f"\n{'='*70}")
        print("🎯 RECHERCHE DE VIDÉOS")
        print(f"{'='*70}\n")
        
        # Récupérer toutes les vidéos de la chaîne
        videos = self.obtenir_videos_chaine(url_chaine)
        
        if not videos:
            return []
        
        # Filtrer par date si spécifié
        if date_debut or date_fin:
            videos = self.filtrer_par_date(videos, date_debut, date_fin)
        
        # Filtrer par mot-clé
        videos_filtrees = self.filtrer_par_mot_cle(
            videos, mot_cle, dans_titre, dans_description
        )
        
        return videos_filtrees
    
    def afficher_resultats(self, videos):
        """
        Affiche les résultats de la recherche
        
        Args:
            videos: Liste des vidéos à afficher
        """
        print(f"\n{'='*70}")
        print(f"📋 RÉSULTATS DE LA RECHERCHE ({len(videos)} vidéos)")
        print(f"{'='*70}\n")
        
        for i, video in enumerate(videos, 1):
            titre = video.get('title', 'N/A')
            date = video.get('upload_date', 'N/A')
            if date != 'N/A' and len(date) == 8:
                date = f"{date[:4]}-{date[4:6]}-{date[6:]}"
            duree = video.get('duration', 0)
            duree_min = duree // 60 if duree else 0
            duree_sec = duree % 60 if duree else 0
            video_id = video.get('id', 'N/A')
            url = f"https://www.youtube.com/watch?v={video_id}"
            
            print(f"{i}. {titre}")
            print(f"   📅 Date: {date}")
            print(f"   ⏱️  Durée: {duree_min}:{duree_sec:02d}")
            print(f"   🔗 URL: {url}")
            print()
    
    def sauvegarder_resultats(self, videos, fichier_sortie="resultats.json"):
        """
        Sauvegarde les résultats dans un fichier JSON
        
        Args:
            videos: Liste des vidéos
            fichier_sortie: Nom du fichier de sortie
        """
        fichier = self.output_dir / fichier_sortie
        
        resultats = []
        for video in videos:
            resultats.append({
                'titre': video.get('title'),
                'url': f"https://www.youtube.com/watch?v={video.get('id')}",
                'date_publication': video.get('upload_date'),
                'duree': video.get('duration'),
                'vues': video.get('view_count'),
                'description': video.get('description', '')[:200] + '...'  # Première partie
            })
        
        with open(fichier, 'w', encoding='utf-8') as f:
            json.dump(resultats, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Résultats sauvegardés dans: {fichier}")
    
    def telecharger_videos_filtrees(self, videos, qualite="best"):
        """
        Télécharge toutes les vidéos filtrées
        
        Args:
            videos: Liste des vidéos à télécharger
            qualite: Qualité de téléchargement
        """
        if not videos:
            print("❌ Aucune vidéo à télécharger")
            return
        
        print(f"\n{'='*70}")
        print(f"📥 TÉLÉCHARGEMENT DE {len(videos)} VIDÉO(S)")
        print(f"{'='*70}\n")
        
        ydl_opts = {
            'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
            'outtmpl': str(self.output_dir / '%(upload_date)s - %(title)s.%(ext)s'),
            'merge_output_format': 'mp4',
        }
        
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            for i, video in enumerate(videos, 1):
                video_id = video.get('id')
                titre = video.get('title', 'N/A')
                
                print(f"\n[{i}/{len(videos)}] Téléchargement de: {titre}")
                
                try:
                    url = f"https://www.youtube.com/watch?v={video_id}"
                    ydl.download([url])
                    print("✅ Téléchargé avec succès!")
                except Exception as e:
                    print(f"❌ Erreur: {e}")



In [5]:

def main():
    """Fonction principale avec exemple d'utilisation"""
    
    # Créer une instance
    searcher = YouTubeChannelSearcher(output_dir="resultats_recherche")
    
    # EXEMPLE D'UTILISATION - Modifiez ces valeurs selon vos besoins
    
    # 1. URL de la chaîne (plusieurs formats possibles)
    url_chaine = "https://www.youtube.com/@FoxNews/videos"

    # 2. Mot-clé à rechercher
    mot_cle = "immigration"
    
    # 3. Plage de dates (format YYYY-MM-DD)
    date_debut = "2024-08-12"  # ou None pour ignorer
    date_fin = "2024-09-25"    # ou None pour ignorer
    
    print("="*70)
    print("🎬 RECHERCHE DE VIDÉOS YOUTUBE")
    print("="*70)
    print(f"\n📺 Chaîne: {url_chaine}")
    print(f"🔑 Mot-clé: '{mot_cle}'")
    if date_debut:
        print(f"📅 Du: {date_debut}")
    if date_fin:
        print(f"📅 Au: {date_fin}")
    
    # DÉCOMMENTEZ LES LIGNES CI-DESSOUS POUR LANCER LA RECHERCHE
    
    
    videos_trouvees = searcher.rechercher_videos(
        url_chaine=url_chaine,
        mot_cle=mot_cle,
        date_debut=date_debut,
        date_fin=date_fin,
        dans_titre=True,        # Rechercher dans le titre
        dans_description=True   # Rechercher dans la description
    )
    
    
    searcher.afficher_resultats(videos_trouvees)
    
    #Sauvegarder les résultats dans un fichier JSON
    searcher.sauvegarder_resultats(videos_trouvees, "resultats.json")
    
    
    print("\n" + "="*70)
    print("✨ INSTRUCTIONS:")
    print("="*70)
    print("1. Modifiez les variables en haut de main():")
    print("   - url_chaine: URL de la chaîne YouTube")
    print("   - mot_cle: Le mot à rechercher")
    print("   - date_debut et date_fin: Plage de dates")
    print("\n2. Décommentez les lignes de code pour lancer la recherche")
    print("\n3. Exécutez le script: python recherche_chaine.py")
    print("="*70)


if __name__ == "__main__":
    main()

🎬 RECHERCHE DE VIDÉOS YOUTUBE

📺 Chaîne: https://www.youtube.com/@FoxNews/videos
🔑 Mot-clé: 'immigration'
📅 Du: 2024-08-12
📅 Au: 2024-09-25

🎯 RECHERCHE DE VIDÉOS

🔍 Récupération des vidéos de la chaîne...


KeyboardInterrupt: 

In [2]:

import yt_dlp

# ========== MODIFIEZ CES DEUX LIGNES ==========
URL_CHAINE = "https://www.youtube.com/@NotJustBikes"
MOT_CLE = "cars"
# ==============================================

print(f"🔍 Recherche de '{MOT_CLE}' sur la chaîne...\n")

# Récupérer toutes les vidéos de la chaîne
url = URL_CHAINE.rstrip('/') + '/videos'
with yt_dlp.YoutubeDL({'quiet': False, 'extract_flat': True}) as ydl:
    info = ydl.extract_info(url, download=False)
    toutes_videos = info.get('entries', [])

print(f"✅ {len(toutes_videos)} vidéos sur la chaîne")

# Filtrer par mot-clé
videos_trouvees = []
for video in toutes_videos:
    if MOT_CLE in video.get('title', ''):
        videos_trouvees.append(video)

print(f"✅ {len(videos_trouvees)} vidéos contiennent '{MOT_CLE}'\n")

# Afficher les résultats
print("=" * 70)
print("RÉSULTATS:")
print("=" * 70 + "\n")

for i, video in enumerate(videos_trouvees, 1):
    print(f"{i}. {video['title']}")
    print(f"   https://www.youtube.com/watch?v={video['id']}\n")


🔍 Recherche de 'cars' sur la chaîne...

[youtube:tab] Extracting URL: https://www.youtube.com/@NotJustBikes/videos
[youtube:tab] @NotJustBikes/videos: Downloading webpage
[download] Downloading playlist: Not Just Bikes - Videos
[youtube:tab] UC0intLFzLaudFG-xAvUEO-A page 1: Downloading API JSON
[youtube:tab] UC0intLFzLaudFG-xAvUEO-A page 2: Downloading API JSON
[youtube:tab] UC0intLFzLaudFG-xAvUEO-A page 3: Downloading API JSON
[youtube:tab] Playlist Not Just Bikes - Videos: Downloading 115 items of 115
[download] Downloading item 1 of 115
[download] Downloading item 2 of 115
[download] Downloading item 3 of 115
[download] Downloading item 4 of 115
[download] Downloading item 5 of 115
[download] Downloading item 6 of 115
[download] Downloading item 7 of 115
[download] Downloading item 8 of 115
[download] Downloading item 9 of 115
[download] Downloading item 10 of 115
[download] Downloading item 11 of 115
[download] Downloading item 12 of 115
[download] Downloading item 13 of 115
[downl

In [3]:
videos_trouvees

[{'title': 'Trams are Great! So why are the Streetcars SO BAD!?',
  'thumbnails': [{'url': 'https://i.ytimg.com/vi/HhQxNHrD6fA/hqdefault.jpg?sqp=-oaymwEbCKgBEF5IVfKriqkDDggBFQAAiEIYAXABwAEG&rs=AOn4CLAmCB1TtZpswg66IKnyCeT1U-eBoA',
    'height': 94,
    'width': 168},
   {'url': 'https://i.ytimg.com/vi/HhQxNHrD6fA/hqdefault.jpg?sqp=-oaymwEbCMQBEG5IVfKriqkDDggBFQAAiEIYAXABwAEG&rs=AOn4CLBoPyEMUqsxjTHrJO-oR3z2LHCeWA',
    'height': 110,
    'width': 196},
   {'url': 'https://i.ytimg.com/vi/HhQxNHrD6fA/hqdefault.jpg?sqp=-oaymwEcCPYBEIoBSFXyq4qpAw4IARUAAIhCGAFwAcABBg==&rs=AOn4CLAQTdrUn9kvJjnODVg9H-0-KQEHsw',
    'height': 138,
    'width': 246},
   {'url': 'https://i.ytimg.com/vi/HhQxNHrD6fA/hqdefault.jpg?sqp=-oaymwEcCNACELwBSFXyq4qpAw4IARUAAIhCGAFwAcABBg==&rs=AOn4CLAsQFT1PlIkiG2VzVvXKUE79zJtCw',
    'height': 188,
    'width': 336}],
  'duration': 3031.0,
  'timestamp': None,
  'ie_key': 'Youtube',
  'id': 'HhQxNHrD6fA',
  '_type': 'url',
  'url': 'https://www.youtube.com/watch?v=HhQxNHrD6fA

In [4]:
toutes_videos

[{'title': 'When Oil Gets Expensive, Cities Get Better',
  'thumbnails': [{'url': 'https://i.ytimg.com/vi/eXNLaHsKMz8/hqdefault.jpg?sqp=-oaymwEbCKgBEF5IVfKriqkDDggBFQAAiEIYAXABwAEG&rs=AOn4CLD3BmORijppiOnoUT6KbUxXXN6-_w',
    'height': 94,
    'width': 168},
   {'url': 'https://i.ytimg.com/vi/eXNLaHsKMz8/hqdefault.jpg?sqp=-oaymwEbCMQBEG5IVfKriqkDDggBFQAAiEIYAXABwAEG&rs=AOn4CLCXPdXt-yBAbkQ4z2NYZqrsHk7UTA',
    'height': 110,
    'width': 196},
   {'url': 'https://i.ytimg.com/vi/eXNLaHsKMz8/hqdefault.jpg?sqp=-oaymwEcCPYBEIoBSFXyq4qpAw4IARUAAIhCGAFwAcABBg==&rs=AOn4CLAaNzYFqCMBtCQ915G_HMyIMlL0Pg',
    'height': 138,
    'width': 246},
   {'url': 'https://i.ytimg.com/vi/eXNLaHsKMz8/hqdefault.jpg?sqp=-oaymwEcCNACELwBSFXyq4qpAw4IARUAAIhCGAFwAcABBg==&rs=AOn4CLDvOHhwbmefd8VRZukpZ5JnFwUAPw',
    'height': 188,
    'width': 336}],
  'duration': 1374.0,
  'timestamp': None,
  'ie_key': 'Youtube',
  'id': 'eXNLaHsKMz8',
  '_type': 'url',
  'url': 'https://www.youtube.com/watch?v=eXNLaHsKMz8',
  '__x

In [5]:
import json
with open("foxnews_videos.json", "w", encoding="utf-8") as fichier:
    json.dump(toutes_videos, fichier, indent=2, ensure_ascii=False)

In [6]:
import pandas as pd

In [8]:
pd.read_json("foxnews_videos.json")

,title,thumbnails,duration,timestamp,ie_key,id,_type,url,__x_forwarded_for_ip
0,"When Oil Gets Expensive, Cities Get Better",[{'url': 'https://i.ytimg.com/vi/eXNLaHsKMz8/h...,1374,NaT,Youtube,eXNLaHsKMz8,url,https://www.youtube.com/watch?v=eXNLaHsKMz8,NaN
1,Every Reason to Hate Cars,[{'url': 'https://i.ytimg.com/vi/umgi-CbaSRU/h...,2911,NaT,Youtube,umgi-CbaSRU,url,https://www.youtube.com/watch?v=umgi-CbaSRU,NaN
2,Every Metro System Should be this Beautiful,[{'url': 'https://i.ytimg.com/vi/WJCCDYbK0Gc/h...,1364,NaT,Youtube,WJCCDYbK0Gc,url,https://www.youtube.com/watch?v=WJCCDYbK0Gc,NaN
3,Why Google Maps Fails in Amsterdam,[{'url': 'https://i.ytimg.com/vi/csHdwHTteOw/h...,1526,NaT,Youtube,csHdwHTteOw,url,https://www.youtube.com/watch?v=csHdwHTteOw,NaN
4,How can a NEW Transit Line be THIS BAD!? (Finc...,[{'url': 'https://i.ytimg.com/vi/Qjp7hqXgInI/h...,1407,NaT,Youtube,Qjp7hqXgInI,url,https://www.youtube.com/watch?v=Qjp7hqXgInI,NaN
...,...,...,...,...,...,...,...,...,...
110,Car-free Streets are Amazing (and we need more...,[{'url': 'https://i.ytimg.com/vi/GlXNVnftaNs/h...,364,NaT,Youtube,GlXNVnftaNs,url,https://www.youtube.com/watch?v=GlXNVnftaNs,NaN
111,Bicycle tour of Frans Halsbuurt - a parking-fr...,[{'url': 'https://i.ytimg.com/vi/Y0ntJ8DM8-8/h...,597,NaT,Youtube,Y0ntJ8DM8-8,url,https://www.youtube.com/watch?v=Y0ntJ8DM8-8,NaN
112,Why Grocery Shopping is Better in Amsterdam,[{'url': 'https://i.ytimg.com/vi/kYHTzqHIngk/h...,227,NaT,Youtube,kYHTzqHIngk,url,https://www.youtube.com/watch?v=kYHTzqHIngk,NaN
113,How Highways Almost Destroyed Amsterdam - Plan...,[{'url': 'https://i.ytimg.com/vi/vI5pbDFDZyI/h...,206,NaT,Youtube,vI5pbDFDZyI,url,https://www.youtube.com/watch?v=vI5pbDFDZyI,NaN


In [17]:
import yt_dlp

# ========== MODIFIEZ CES DEUX LIGNES ==========
URL_CHAINE = "https://www.youtube.com/@FoxNews"
MOT_CLE = "immigration"
# ==============================================

print(f"Recherche de '{MOT_CLE}' sur la chaine...\n")

url = URL_CHAINE.rstrip('/') + '/videos'

# Passe 1 : liste rapide
with yt_dlp.YoutubeDL({'quiet': True, 'extract_flat': True}) as ydl:
    info = ydl.extract_info(url, download=False)
    toutes_videos = info.get('entries', [])

print(f"{len(toutes_videos)} videos sur la chaine")

videos_filtrees = [v for v in toutes_videos if MOT_CLE.lower() in v.get('title', '').lower()]

print(f"{len(videos_filtrees)} videos contiennent '{MOT_CLE}'\n")

# Passe 2 : details complets uniquement sur les videos filtrees
opts = {'quiet': True, 'extract_flat': False}
FoxNews_results = []

with yt_dlp.YoutubeDL(opts) as ydl:
    for v in videos_filtrees:
        detail = ydl.extract_info(f"https://www.youtube.com/watch?v={v['id']}", download=False)
        FoxNews_results.append({
            'channel': detail.get('channel'),
            'title': detail.get('title'),
            'upload_date': detail.get('upload_date'),  # format YYYYMMDD
            'url': f"https://www.youtube.com/watch?v={v['id']}"
        })

print("=" * 70)
print("RESULTATS:")
print("=" * 70 + "\n")

for i, v in enumerate(FoxNews_results, 1):
    print(f"{i}. {v['title']}")
    print(f"   Date : {v['upload_date']}")
    print(f"   {v['url']}\n")

Recherche de 'immigration' sur la chaine...



KeyboardInterrupt: 

In [13]:
import pandas as pd

df = pd.DataFrame(resultats)
print(df)

                                               title         channel  \
0                          Every Reason to Hate Cars  Not Just Bikes   
1  Trams are Great! So why are the Streetcars SO ...  Not Just Bikes   
2  How Self-Driving Cars will Destroy Cities (and...  Not Just Bikes   
3  Amsterdam Closed This Bridge to Cars (but not ...  Not Just Bikes   
4  Germany's "Green" City (with more bikes than c...  Not Just Bikes   
5                   How Toronto Got Addicted to Cars  Not Just Bikes   
6                  Cities Aren't Loud: Cars Are Loud  Not Just Bikes   
7               Rotterdam: the City Rebuilt for Cars  Not Just Bikes   
8               The Miniature Microcars of Amsterdam  Not Just Bikes   
9  Why Cars Rarely Crash into Buildings in the Ne...  Not Just Bikes   

  upload_date                                          url  
0    20260329  https://www.youtube.com/watch?v=umgi-CbaSRU  
1    20250720  https://www.youtube.com/watch?v=HhQxNHrD6fA  
2    20241110  https://w

In [14]:
df.to_csv('resultats.csv', index=False)